# Fase 3b: Selección de modelos
---
Este cuaderno evalúa el rendimiento de cuatro modelos preentrenados  especializados en análisis de sentimientos, emociones y clasificación temática. 

El objetivo es determinar cuáles de ellos logran captar mejor las sutilezas poéticas, el sarcasmo y el contexto lírico del caso de estudio; ya que antes de decidir que modelo utilizar para la clasificación de emociones, es necesario realizar una fase de *benchmarking* cualitativo.

In [1]:
import pandas as pd
from transformers import pipeline

In [2]:
import warnings
warnings.filterwarnings('ignore')

### 3b.1 Definición de los candidatos
Para la evaluación, se seleccionaron cuatro enfoques arquitectónicos distintos:
1. **`j-hartmann/emotion-english-distilroberta-base`**: Clasificación basada en la teoría clásica de Paul Ekman (7 emociones básicas). Útil para emociones primarias, pero potencialmente limitado.
2. **`samlowe/roberta-base-go_emotions`**: Basado en el dataset de Google GoEmotions, capaz de detectar 28 emociones cognitivas sutiles.
3. **`cardiffnlp/twitter-roberta-base-sentiment-latest`**: Entrenado con millones de tuits, por lo que pilla muy bien el sarcasmo y el lenguaje moderno.
4. **`facebook/bart-large-mnli`**: Un modelo *Zero-Shot Classification*. No está atado a etiquetas predefinidas, lo que permite inferir temáticas personalizadas sin necesidad de *fine-tuning*.

In [3]:
clf_hartmann = pipeline('text-classification', model='j-hartmann/emotion-english-distilroberta-base')
clf_goemotions = pipeline('text-classification', model='samlowe/roberta-base-go_emotions')
clf_sentiment = pipeline('text-classification', model='cardiffnlp/twitter-roberta-base-sentiment-latest')

clf_zeroshot = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')
personalized_labels = ['romance', 'heartbreak', 'nostalgia', 'empowerment', 'self-reflection', 'celebration', 'melancholy', 'revenge', 'anxiety']

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

### 3b.2 Metodología de evaluación y ejecución
Para no sobrecargar la inferencia y respetar las ventanas de contexto de los Transformers (habitualmente de 512 *tokens*), se truncaron las letras a los primeros 600 caracteres. Para garantizar la variedad temática en la prueba, se iteró extrayendo únicamente la primera canción de cada álbum del catálogo.

En cada iteración, el texto fue procesado simultáneamente por los cuatro modelos para comparar sus predicciones cara a cara.

In [6]:
df = pd.read_csv('../data/processed/taylor_swift_processed.csv')
df_album = df.groupby('album').first().reset_index()

for idx, row in df_album.iterrows():
    album = row['album']
    title = row['title']
    full_lyrics = str(row['lyrics_full'])

    lyrics_clean = full_lyrics.replace('\n', ' ')
    lyrics_cropped = lyrics_clean[:600]

    print("="*80)
    print(f"Album: {album} | Song: {title}")
    print(f"Lyrics: '{lyrics_cropped}...'")
    print("-"*80)

    res_hart = clf_hartmann(lyrics_cropped)[0]
    print(f"Modelo: j-hartmann/emotion-english-distilroberta-base\n {res_hart['label'].upper()} ({res_hart['score']:.2f})")

    res_go = clf_goemotions(lyrics_cropped, top_k=2)
    print(f"Modelo: samlowe/roberta-base-go_emotions\n 1º {res_go[0]['label'].upper()} ({res_go[0]['score']:.2f}) | 2º {res_go[1]['label'].upper()} ({res_go[1]['score']:.2f})")

    res_sent = clf_sentiment(lyrics_cropped)[0]
    print(f"Modelo: cardiffnlp/twitter-roberta-base-sentiment-latest\n {res_sent['label'].upper()} ({res_sent['score']:.2f})")

    res_zs = clf_zeroshot(lyrics_cropped, candidate_labels=personalized_labels)
    print(f"Modelo: facebook/bart-large-mnli\n 1º {res_zs['labels'][0].upper()} ({res_zs['scores'][0]:.2f}) | 2º {res_zs['labels'][1].upper()} ({res_zs['scores'][1]:.2f})")
    print("="*80 + "\n")

Album: 1989 (Taylor’s Version) | Song: Wildest Dreams (Taylor’s Version)
Lyrics: ' He said, "Let's get out of this town Drive out of the city, away from the crowds" I thought, "Heaven can't help me now" Nothing lasts forever But this is gonna take me down He's so tall and handsome as hell He's so bad, but he does it so well I can see the end as it begins My one condition is Say you'll remember me Standin' in a nice dress Starin' at the sunset, babe Red lips and rosy cheeks Say you'll see me again Even if it's just in your Wildest dreams, ah, ha Wildest dreams, ah, ha I said, "No one has to know what wе do" His hands are in my hair, his clothes are in my room And his voicе i...'
--------------------------------------------------------------------------------
Modelo: j-hartmann/emotion-english-distilroberta-base
 NEUTRAL (0.52)
Modelo: samlowe/roberta-base-go_emotions
 1º ADMIRATION (0.74) | 2º NEUTRAL (0.08)
Modelo: cardiffnlp/twitter-roberta-base-sentiment-latest
 NEUTRAL (0.41)
Modelo

### Conclusión del benchmark
Tras el análisis cualitativo de los resultados, se tomaron las siguientes decisiones de diseño para el sistema final:
* `samlowe/roberta-base-go_emotions` superó a `j-hartmann/emotion-english-distilroberta-base`, ya que mientras `j-hartmann/emotion-english-distilroberta-base` tendía a encasillar todas las baladas en "Tristeza", `samlowe/roberta-base-go_emotions` logró diferenciar matices como "Decepción" o "Admiración", aportando mucha más riqueza.
* Por el lado de las temáticas, `facebook/bart-large-mnli` demostró una precisión excepcional adaptándose al diccionario de temas que se le proporcionó específicamente para el caso de estudio, convirtiéndose en la herramienta principal para el modelado de temas.
* El modelo de `cardiffnlp/twitter-roberta-base-sentiment-latest` fue descartado para la métrica principal, siendo sustituido por polaridad clásica con *TextBlob* para ahorrar tiempos de computación, dejando que los Transformers pesados se centrasen únicamente en las emociones y temáticas.